In [ ]:
## Notebook for generating LowRes, LowResResp and LowResRespBand training data from HighRes data

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import numpy as np
from tqdm import tqdm
from process_utils import *
import nibabel as nib

In [ ]:
#Makes directory for training data

if not os.path.exists('./training_data'):
    os.makedirs('./training_data')

In [ ]:
# For MMWHS data
mmwhs_number = 1 #Pick how many datasets to process
for pat in tqdm(range(mmwhs_number)): 
	xyz = lowest_point_along_y(f"/workspaces/Desktop/data_process/112_seg_data_new/mmwhs/High_Res_MMWHS_NoAug_{pat}_masks.nii.gz")
	target = np.load(f'/workspaces/Desktop/data_process/112_seg_data_new/mmwhs/High_Res_MMWHS_NoAug_{pat}_img.npy')
	thick_slice =[]
	y_def = []
	x_def = []
	y_def_max = []
	x_def_max = []
	for i in range(28):
		combined = target[i*4:(i+1)*4,:,:]
		combined = np.mean(combined,axis=0)
		thick_slice.append(combined)
	thick_slice = np.array(thick_slice)
	aug_num=50
	for aug in range(aug_num): 

		resp_def , deformed_downsampled_vols = (respiratory_deformations([thick_slice],xyz)) #Applies respiratory deformations to low-res data
		banded = add_bands(deformed_downsampled_vols[0]) #Applies contrast changes to repiratory artefacted data
  
		np.save(f'training_data/Low_Res_small_resp_MMWHS_{pat*aug_num + aug}.npy', norm(deformed_downsampled_vols[0]))
		np.save(f'./training_data/Low_Res_resp_band_MMWHS_{pat*aug_num + aug}.npy', norm(banded))
	np.save(f'./training_data/Low_Res_MMWHS_{pat*aug_num}.npy', norm(thick_slice))
	np.save(f'training_data/High_Res_MMWHS_{pat*aug_num}.npy', norm(target))

In [ ]:
# For HVSMR data
hvsmr_number = 60 #Pick how many datasets to process
for pat in tqdm(range(hvsmr_number)): 
	xyz = lowest_point_along_y(f"/workspaces/Desktop/data_process/112_seg_data_new/hvsmr/High_Res_HVSMR_NoAug_{pat}_masks.nii.gz")
	target = np.load(f'/workspaces/Desktop/data_process/112_seg_data_new/hvsmr/High_Res_HVSMR_NoAug_{pat}_img.npy')
	thick_slice =[]
	for i in range(28):
		combined = target[i*4:(i+1)*4,:,:]
		combined = np.mean(combined,axis=0)
		thick_slice.append(combined)
	thick_slice = np.array(thick_slice)
	aug_num = 50
	for aug in range(aug_num): 

		resp_def , deformed_downsampled_vols = (respiratory_deformations([thick_slice],xyz)) #Applies respiratory deformations to low-res data
		banded = add_bands(deformed_downsampled_vols[0]) #Applies contrast changes to repiratory artefacted data
  
		np.save(f'./training_data//Low_Res_small_resp_HVSMR_{pat*aug_num + aug}.npy', norm(deformed_downsampled_vols[0]))
		np.save(f'./training_data/Low_Res_resp_band_HVSMR_{pat*aug_num + aug}.npy', norm(banded))
	np.save(f'./training_data/Low_Res_HVSMR_{pat*aug_num}.npy', norm(thick_slice))
	np.save(f'./training_data//High_Res_HVSMR_{pat*aug_num}.npy', norm(target))